<h1 align='center'> 영상처리 프로그래밍 실습 9</h1>

<h6 align='right'> 2024. 5. 29. </h6>

<div class="alert alert-block alert-info">
    
- 파일 이름에서 00000000을 자신의 학번으로, name을 자신의 이름으로 수정하세요.

- 다음 줄에 자신의 이름, 학번, 학과(전공)을 적으세요.

* 이름: 이한석  &nbsp;&nbsp;          학번: 20215226   &nbsp;&nbsp;         학과(전공): 빅데이터학과
    
</div>

- JupyterLab 문서의 최신 버전은 [JupyterLab Documentation](https://jupyterlab.readthedocs.io/en/stable/index.html#/)을  참고하라

- Markdown은 [Markdown Guide](https://www.markdownguide.org/)를 참고하라.
- [Markdown Cheat Sheet](https://www.markdownguide.org/cheat-sheet/)

* 제출 마감: 6월 4일 (화) 오후 10:00까지 최종본을 SmartLEAD제출


In [1]:
import cv2
import matplotlib.pyplot as plt

import numpy as np
print("OpenCV version", cv2.__version__)
print("NumPy version", np.__version__)

OpenCV version 4.11.0
NumPy version 2.0.2


### 문제 1. 

지금까지 만들고 있는 영상처리 프로그램에 다음 기능을 추가하라.
- 다양한 에지 검출 기능
- 가우시안 잡음 추가 기능
- 영상 회전 기능

In [20]:
from PyQt5.QtWidgets import (
    QApplication, QMainWindow, QLabel, QSlider,
    QWidget, QHBoxLayout, QVBoxLayout, QSizePolicy,
    QAction, QFileDialog
)
from PyQt5.QtCore import Qt
from PyQt5.QtGui import QPixmap, QImage, QFont
import numpy as np
import cv2
import os
import sys

class MovingAverageViewer(QMainWindow):
    def __init__(self, filename):
        super().__init__()
        self.setWindowTitle("PyQt 이미지 필터링")
        self.setGeometry(200, 200, 1200, 600)

        self.original_image = None
        self.filtered_image = None
        self.current_image = None  # 현재 보여주는 이미지 (원본 or 잡음 추가 or 회전된 상태 등)

        self.rotation_angle = 0  # 0, 90, 180, 270 (시계방향 회전 각도)

        # 이미지 표시용 QLabel
        self.orig_label = QLabel()
        self.orig_label.setSizePolicy(QSizePolicy.Ignored, QSizePolicy.Ignored)
        self.orig_label.setAlignment(Qt.AlignCenter)

        self.filtered_label = QLabel()
        self.filtered_label.setSizePolicy(QSizePolicy.Ignored, QSizePolicy.Ignored)
        self.filtered_label.setAlignment(Qt.AlignCenter)

        # 커널 사이즈 표시용 텍스트
        self.kernel_label = QLabel("Kernel Size: 3")
        self.kernel_label.setFont(QFont("Arial", 12))

        # 슬라이더
        self.slider = QSlider(Qt.Horizontal)
        self.slider.setMinimum(1)
        self.slider.setMaximum(51)
        self.slider.setValue(3)
        self.slider.valueChanged.connect(self.apply_filter)

        # 메뉴 바 생성
        self.create_menu()

        # 레이아웃 설정
        slider_layout = QHBoxLayout()
        slider_layout.addWidget(QLabel("Kernel:"))
        slider_layout.addWidget(self.slider)
        slider_layout.addWidget(self.kernel_label)

        image_layout = QHBoxLayout()
        image_layout.addWidget(self.orig_label)
        image_layout.addWidget(self.filtered_label)

        main_layout = QVBoxLayout()
        main_layout.addLayout(image_layout)
        main_layout.addLayout(slider_layout)

        central_widget = QWidget()
        central_widget.setLayout(main_layout)
        self.setCentralWidget(central_widget)

        # 초기 이미지 로드
        self.load_image(filename)

    def create_menu(self):
        menubar = self.menuBar()

        # 파일 메뉴
        file_menu = menubar.addMenu("파일")
        open_action = QAction("열기", self)
        open_action.triggered.connect(self.open_file_dialog)
        file_menu.addAction(open_action)

        # 필터 메뉴
        filter_menu = menubar.addMenu("필터")

        mean_filter_action = QAction("이동 평균 필터", self)
        mean_filter_action.triggered.connect(lambda: self.apply_filter("mean"))
        filter_menu.addAction(mean_filter_action)

        gaussian_filter_action = QAction("가우시안 필터", self)
        gaussian_filter_action.triggered.connect(lambda: self.apply_filter("gaussian"))
        filter_menu.addAction(gaussian_filter_action)

        median_filter_action = QAction("미디언 필터", self)
        median_filter_action.triggered.connect(lambda: self.apply_filter("median"))
        filter_menu.addAction(median_filter_action)

        box_filter_action = QAction("박스 필터", self)
        box_filter_action.triggered.connect(lambda: self.apply_filter("box"))
        filter_menu.addAction(box_filter_action)

        # 에지 검출 메뉴
        edge_menu = menubar.addMenu("에지 검출")

        roberts_action = QAction("로버츠", self)
        roberts_action.triggered.connect(lambda: self.apply_filter("roberts"))
        edge_menu.addAction(roberts_action)

        prewitt_action = QAction("프리윗", self)
        prewitt_action.triggered.connect(lambda: self.apply_filter("prewitt"))
        edge_menu.addAction(prewitt_action)

        sobel_action = QAction("소벨", self)
        sobel_action.triggered.connect(lambda: self.apply_filter("sobel"))
        edge_menu.addAction(sobel_action)

        laplacian_action = QAction("라플라시안", self)
        laplacian_action.triggered.connect(lambda: self.apply_filter("laplacian"))
        edge_menu.addAction(laplacian_action)

        # 추가 기능 메뉴
        extra_menu = menubar.addMenu("추가 기능")

        noise_action = QAction("가우시안 잡음 추가", self)
        noise_action.triggered.connect(self.add_gaussian_noise)
        extra_menu.addAction(noise_action)

        rotate_action = QAction("영상 90도 회전", self)
        rotate_action.triggered.connect(self.rotate_image)
        extra_menu.addAction(rotate_action)

    def open_file_dialog(self):
        filename, _ = QFileDialog.getOpenFileName(self, "이미지 열기", "", "Image Files (*.png *.jpg *.bmp)")
        if filename:
            self.load_image(filename)
            self.rotation_angle = 0  # 초기화
            self.current_image = self.original_image.copy()
            self.apply_filter(self.slider.value())

    def load_image(self, filename):
        if not os.path.exists(filename):
            print(f"[오류] 파일 '{filename}'이 존재하지 않습니다.")
            return

        image = cv2.imread(filename)
        if image is None:
            print(f"[오류] 이미지를 불러올 수 없습니다: {filename}")
            return

        self.original_image = image
        self.current_image = self.original_image.copy()
        self.rotation_angle = 0
        self.apply_filter(self.slider.value())

    def add_gaussian_noise(self):
        if self.current_image is None:
            return

        image = self.current_image.copy().astype(np.float32) / 255.0

        mean = 0
        sigma = 0.05
        gauss = np.random.normal(mean, sigma, image.shape)
        noisy = image + gauss
        noisy = np.clip(noisy, 0, 1)
        noisy = (noisy * 255).astype(np.uint8)

        self.current_image = noisy
        self.apply_filter(self.slider.value())

    def rotate_image(self):
        if self.current_image is None:
            return

        self.rotation_angle = (self.rotation_angle + 90) % 360
        # OpenCV rotate code
        if self.rotation_angle == 90:
            rotated = cv2.rotate(self.current_image, cv2.ROTATE_90_CLOCKWISE)
        elif self.rotation_angle == 180:
            rotated = cv2.rotate(self.current_image, cv2.ROTATE_180)
        elif self.rotation_angle == 270:
            rotated = cv2.rotate(self.current_image, cv2.ROTATE_90_COUNTERCLOCKWISE)
        else:  # 0도면 원본으로
            rotated = self.original_image.copy()

        self.current_image = rotated
        self.apply_filter(self.slider.value())

    def apply_filter(self, val):
        kernel = self.slider.value()
        if kernel % 2 == 0:
            kernel -= 1  # 커널은 홀수

        self.kernel_label.setText(f"{kernel}")  # 커널 크기 표시

        if self.current_image is None:
            return

        # 필터 선택
        if isinstance(val, int):  # 슬라이더 값일 경우 기본 이동 평균 필터
            filter_type = "mean"
        else:
            filter_type = val

        img = self.current_image

        if filter_type == "mean":
            filtered = cv2.blur(img, (kernel, kernel))
        elif filter_type == "gaussian":
            filtered = cv2.GaussianBlur(img, (kernel, kernel), 0)
        elif filter_type == "median":
            filtered = cv2.medianBlur(img, kernel)
        elif filter_type == "box":
            filtered = cv2.boxFilter(img, -1, (kernel, kernel))
        elif filter_type == "roberts":
            filtered = self.roberts_edge(img)
        elif filter_type == "prewitt":
            filtered = self.prewitt_edge(img)
        elif filter_type == "sobel":
            filtered = self.sobel_edge(img)
        elif filter_type == "laplacian":
            filtered = self.laplacian_edge(img)
        else:
            filtered = img

        self.filtered_image = filtered

        self.orig_label.setPixmap(self.cv2_to_pixmap(self.original_image).scaled(
            self.orig_label.size(), Qt.KeepAspectRatio, Qt.SmoothTransformation))
        self.filtered_label.setPixmap(self.cv2_to_pixmap(filtered).scaled(
            self.filtered_label.size(), Qt.KeepAspectRatio, Qt.SmoothTransformation))

    def roberts_edge(self, img):
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        kernelx = np.array([[1, 0], [0, -1]])
        kernely = np.array([[0, 1], [-1, 0]])
        x = cv2.filter2D(gray, -1, kernelx)
        y = cv2.filter2D(gray, -1, kernely)
        edge = cv2.convertScaleAbs(x + y)
        return cv2.cvtColor(edge, cv2.COLOR_GRAY2BGR)

    def prewitt_edge(self, img):
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        kernelx = np.array([[1, 0, -1], [1, 0, -1], [1, 0, -1]])
        kernely = np.array([[1, 1, 1], [0, 0, 0], [-1, -1, -1]])
        x = cv2.filter2D(gray, -1, kernelx)
        y = cv2.filter2D(gray, -1, kernely)
        edge = cv2.convertScaleAbs(x + y)
        return cv2.cvtColor(edge, cv2.COLOR_GRAY2BGR)

    def sobel_edge(self, img):
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        x = cv2.Sobel(gray, cv2.CV_16S, 1, 0)
        y = cv2.Sobel(gray, cv2.CV_16S, 0, 1)
        absX = cv2.convertScaleAbs(x)
        absY = cv2.convertScaleAbs(y)
        edge = cv2.addWeighted(absX, 0.5, absY, 0.5, 0)
        return cv2.cvtColor(edge, cv2.COLOR_GRAY2BGR)

    def laplacian_edge(self, img):
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        edge = cv2.Laplacian(gray, cv2.CV_16S, ksize=3)
        edge = cv2.convertScaleAbs(edge)
        return cv2.cvtColor(edge, cv2.COLOR_GRAY2BGR)

    def resizeEvent(self, event):
        super().resizeEvent(event)
        self.apply_filter(self.slider.value())

    def cv2_to_pixmap(self, img):
        rgb_image = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w, ch = rgb_image.shape
        bytes_per_line = ch * w
        qimage = QImage(rgb_image.data, w, h, bytes_per_line, QImage.Format_RGB888)
        return QPixmap.fromImage(qimage)
    
# 실행 함수
def moving_average_filtering_pyqt(filename):
    app = QApplication.instance()
    if app is None:
        app = QApplication(sys.argv)
    viewer = MovingAverageViewer(filename)
    viewer.show()
    app.exec_()


moving_average_filtering_pyqt("bird.jpg")